In [6]:
pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to C:\Users\KRITHI S KULAL\AppData\Local\Temp\pip-req-build-7tu6l6k3
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36876 sha256=4c63f975834c739540cd82e04d1125084292ed6993ad08b479071c799000d85b
  Stored in directory: C:\Users\KRITHI S KULAL\AppData\Local\Temp\pip-ephem-wheel-cache-9_ahi_6u\wheels\15\d7\bd\05f5f23b7dcbe70cbc6783b06f12143b0cf1a5da5c7b52dcc5
Successfully built segment_anything
Note: you may need to restart the kernel to use

  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git 'C:\Users\KRITHI S KULAL\AppData\Local\Temp\pip-req-build-7tu6l6k3'


In [7]:
pip install opencv-python matplotlib tqdm

  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl (40.2 MB)
Note: you may need to restart the kernel to use updated packages.


In [8]:
from segment_anything import sam_model_registry

print("SAM Installed")

SAM Installed


In [11]:
from pathlib import Path

for file in Path("../").rglob("*vit*.pth"):
    print(file)

..\checkpoints\sam_vit_b_01ec64.pth


In [12]:
import torch
import numpy as np

from pathlib import Path
from tqdm import tqdm

from segment_anything import sam_model_registry

In [13]:
DEVICE = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", DEVICE)

Device: cuda


In [14]:
PROJECT_ROOT = Path("../")

TRAIN_IMAGE_DIR = (
    PROJECT_ROOT /
    "dataset/processed/STRUM/train/images"
)

VAL_IMAGE_DIR = (
    PROJECT_ROOT /
    "dataset/processed/STRUM/val/images"
)

TEST_IMAGE_DIR = (
    PROJECT_ROOT /
    "dataset/processed/STRUM/test/images"
)

SAM_CHECKPOINT = (
    PROJECT_ROOT /
    "checkpoints/sam_vit_b_01ec64.pth"
)

EMBEDDING_ROOT = (
    PROJECT_ROOT /
    "dataset/sam_embeddings/strum_rgb_vit_b"
)

TRAIN_EMBED_DIR = (
    EMBEDDING_ROOT /
    "train"
)

VAL_EMBED_DIR = (
    EMBEDDING_ROOT /
    "val"
)

TEST_EMBED_DIR = (
    EMBEDDING_ROOT /
    "test"
)

TRAIN_EMBED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

VAL_EMBED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

TEST_EMBED_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [17]:
sam = sam_model_registry[
    "vit_b"
](
    checkpoint=str(SAM_CHECKPOINT)
)

sam.to(DEVICE)

sam.eval()

print("SAM Loaded")

SAM Loaded


In [19]:
train_images = sorted(
    TRAIN_IMAGE_DIR.glob("*.npy")
)

val_images = sorted(
    VAL_IMAGE_DIR.glob("*.npy")
)

test_images = sorted(
    TEST_IMAGE_DIR.glob("*.npy")
)

print(
    "Train:",
    len(train_images)
)

print(
    "Val:",
    len(val_images)
)

print(
    "Test:",
    len(test_images)
)

Train: 2140
Val: 267
Test: 268


In [23]:
print(sample_image.shape)
print(sample_image.min())
print(sample_image.max())

(3, 128, 128)
0.0
1.0


In [24]:
image_tensor = torch.tensor(
    sample_image,
    dtype=torch.float32
).unsqueeze(0)

image_tensor = image_tensor.to(DEVICE)

print(image_tensor.shape)

torch.Size([1, 3, 128, 128])


In [27]:
import torch.nn.functional as F

image_tensor = torch.tensor(
    sample_image,
    dtype=torch.float32
).unsqueeze(0)

image_tensor = F.interpolate(
    image_tensor,
    size=(1024, 1024),
    mode="bilinear",
    align_corners=False
)

image_tensor = image_tensor.to(DEVICE)

print(image_tensor.shape)

torch.Size([1, 3, 1024, 1024])


In [28]:
with torch.no_grad():

    embedding = sam.image_encoder(
        image_tensor
    )

print(embedding.shape)

torch.Size([1, 256, 64, 64])


In [29]:
import torch.nn.functional as F

def extract_embedding(image_path):

    image = np.load(image_path)

    image_tensor = torch.tensor(
        image,
        dtype=torch.float32
    ).unsqueeze(0)

    image_tensor = F.interpolate(
        image_tensor,
        size=(1024, 1024),
        mode="bilinear",
        align_corners=False
    )

    image_tensor = image_tensor.to(DEVICE)

    with torch.no_grad():

        embedding = sam.image_encoder(
            image_tensor
        )

    embedding = (
        embedding
        .squeeze(0)
        .cpu()
        .numpy()
        .astype(np.float32)
    )

    return embedding

In [30]:
embedding = extract_embedding(
    train_images[0]
)

print(embedding.shape)

print(embedding.dtype)

print(np.isnan(embedding).sum())

(256, 64, 64)
float32
0


In [31]:
test_path = TRAIN_EMBED_DIR / "test_embedding.npy"

np.save(
    test_path,
    embedding
)

print(test_path)

..\dataset\sam_embeddings\strum_rgb_vit_b\train\test_embedding.npy


In [32]:
loaded = np.load(
    test_path
)

print(loaded.shape)

(256, 64, 64)


In [33]:
def process_split(
    image_files,
    output_dir,
    split_name
):

    print(
        f"\nProcessing {split_name}..."
    )

    for image_path in tqdm(
        image_files
    ):

        output_path = (
            output_dir /
            image_path.name
        )

        if output_path.exists():
            continue

        embedding = extract_embedding(
            image_path
        )

        np.save(
            output_path,
            embedding
        )

    print(
        f"{split_name} completed."
    )

In [34]:
process_split(
    train_images,
    TRAIN_EMBED_DIR,
    "TRAIN"
)


Processing TRAIN...


100%|██████████| 2140/2140 [31:51<00:00,  1.12it/s] 

TRAIN completed.


In [35]:
process_split(
    val_images,
    VAL_EMBED_DIR,
    "VAL"
)


Processing VAL...


100%|██████████| 267/267 [03:45<00:00,  1.19it/s]

VAL completed.


In [36]:
process_split(
    test_images,
    TEST_EMBED_DIR,
    "TEST"
)


Processing TEST...


100%|██████████| 268/268 [03:28<00:00,  1.29it/s]

TEST completed.


In [37]:
train_embed_count = len(
    list(
        TRAIN_EMBED_DIR.glob("*.npy")
    )
)

val_embed_count = len(
    list(
        VAL_EMBED_DIR.glob("*.npy")
    )
)

test_embed_count = len(
    list(
        TEST_EMBED_DIR.glob("*.npy")
    )
)

print(
    "Train:",
    train_embed_count
)

print(
    "Val:",
    val_embed_count
)

print(
    "Test:",
    test_embed_count
)

Train: 2141
Val: 267
Test: 268


In [38]:
train_image_names = set(
    f.name for f in train_images
)

train_embedding_names = set(
    f.name
    for f in TRAIN_EMBED_DIR.glob("*.npy")
)

extra_files = (
    train_embedding_names
    - train_image_names
)

print(extra_files)

{'test_embedding.npy'}


In [39]:
extra_file = (
    TRAIN_EMBED_DIR /
    "test_embedding.npy"
)

if extra_file.exists():

    extra_file.unlink()

    print(
        "Deleted:",
        extra_file.name
    )

Deleted: test_embedding.npy


In [40]:
train_embed_count = len(list(TRAIN_EMBED_DIR.glob("*.npy")))
val_embed_count = len(list(VAL_EMBED_DIR.glob("*.npy")))
test_embed_count = len(list(TEST_EMBED_DIR.glob("*.npy")))

print("Train:", train_embed_count)
print("Val:", val_embed_count)
print("Test:", test_embed_count)

Train: 2140
Val: 267
Test: 268


## Notebook Summary – SAM Feature Extraction

This notebook generates and stores **Segment Anything Model (SAM) image embeddings** for the preprocessed STRUM Sentinel-2 dataset. Instead of extracting features during model training, SAM embeddings are precomputed and cached, significantly reducing training time for subsequent Hybrid SAM-U-Net experiments.

### Objective

The primary goal of this notebook is to:

* Load a pretrained SAM ViT-B model.
* Convert RGB flood-mapping images into SAM-compatible inputs.
* Extract high-level image embeddings using the SAM image encoder.
* Save embeddings to disk for efficient reuse during model training.

### SAM Configuration

| Property             | Value                  |
| -------------------- | ---------------------- |
| SAM Variant          | ViT-B                  |
| Checkpoint           | `sam_vit_b_01ec64.pth` |
| Device               | CUDA (GPU)             |
| Input Image Size     | 128 × 128              |
| SAM Input Resolution | 1024 × 1024            |
| Embedding Dimension  | 256                    |
| Embedding Size       | 64 × 64                |

### Input and Feature Shapes

| Stage                | Shape              |
| -------------------- | ------------------ |
| Original RGB Image   | (3, 128, 128)      |
| Tensor Batch         | (1, 3, 128, 128)   |
| Resized SAM Input    | (1, 3, 1024, 1024) |
| SAM Embedding Output | (1, 256, 64, 64)   |
| Saved Embedding      | (256, 64, 64)      |

### Dataset Processed

| Split      |   Samples |
| ---------- | --------: |
| Train      |     2,140 |
| Validation |       267 |
| Test       |       268 |
| **Total**  | **2,675** |

### Feature Extraction Statistics

| Split      | Samples Processed | Processing Time |
| ---------- | ----------------: | --------------- |
| Train      |             2,140 | 31 min 51 sec   |
| Validation |               267 | 3 min 45 sec    |
| Test       |               268 | 3 min 28 sec    |

### Embedding Characteristics

| Property                | Value   |
| ----------------------- | ------- |
| Data Type               | float32 |
| Channels                | 256     |
| Spatial Resolution      | 64 × 64 |
| Missing Values Detected | 0       |

### Data Integrity Verification

After feature extraction:

| Check                        | Result               |
| ---------------------------- | -------------------- |
| Train Embeddings             | 2,140                |
| Validation Embeddings        | 267                  |
| Test Embeddings              | 268                  |
| Unexpected File Found        | `test_embedding.npy` |
| Action Taken                 | Removed              |
| Final Dataset Count Verified | ✓                    |

### Workflow Summary

1. Loaded the pretrained SAM ViT-B checkpoint.
2. Loaded normalized RGB images from the STRUM dataset.
3. Resized images to SAM's required input resolution (1024 × 1024).
4. Passed images through the SAM image encoder.
5. Generated 256-channel feature embeddings.
6. Saved embeddings as NumPy files.
7. Verified embedding dimensions and dataset counts.
8. Removed accidental duplicate files and revalidated the dataset.

## Conclusion

The SAM feature extraction pipeline was successfully completed for the entire STRUM dataset. Each RGB image was transformed into a compact feature representation of size **256 × 64 × 64** using the pretrained SAM ViT-B image encoder. A total of **2,675 embeddings** were generated and organized across the training, validation, and test splits. The resulting cached embeddings provide rich semantic information while eliminating the need for repeated SAM inference during Hybrid SAM-U-Net training, enabling faster experimentation and reduced computational cost.
